# Argus — train the weapon detection model (version 3)

**Version 3** = version 2 (clean Open Images labels only) + "bad footage" training (step 6b) + every photo used only once (no copies leaking between training and checking). Results still go to `argus_weapons/runs/weapons_v2`.

This notebook trains **our own weapon detector** for the Argus AI pipeline, with four classes:

| Class | Covers |
|---|---|
| `handgun` | pistols, revolvers |
| `long_gun` | rifles (incl. AK-style), shotguns, machine guns |
| `knife` | knives, daggers |
| `machete` | long blades: machetes, pangas, swords |

**How to run it (about 4–5 hours, almost all of it waiting):**

*On Kaggle instead of Colab* (free, ~30 GPU hours/week — handy when Colab says you've hit its usage limit): kaggle.com → **Code → New Notebook → File → Import Notebook** (choose this file) → right-hand **Settings**: *Accelerator* **GPU T4 x2**, *Internet* **On** (needs a verified phone number) → **Run All**. Keep the tab open; at the end download `argus_weapons/best.pt` from the **Output** panel. The steps below are for Colab.

**Every new session restarts once:** step 2 installs the tools and then restarts the session (Colab says it "crashed/restarted" — that's intended). Just press **Run all** again; the second time it goes straight through.

**Save GPU time — prepare the data first, without a GPU (recommended):**
* **Run 1 (no GPU):** Runtime → Change runtime type → **CPU** → Save → **Run all**. Steps 2–6b download and prepare the pictures (about 1–2 hours) and save them to Google Drive as `argus_weapons/weapons_data_v2.zip`. It then stops at step 8 with "DATA IS READY". This uses no GPU allowance.
* **Run 2 (GPU):** Runtime → Change runtime type → **T4 GPU** → Save → **Run all**. It restores the pictures from Drive in a few minutes and goes straight to training. If the GPU time runs out, Run all again another day: training resumes where it stopped.

The steps below describe a single run with the GPU from the start, which also works but spends GPU time on downloads.

1. In Colab: **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all**. Allow access to Google Drive when asked (the model is saved there, so nothing is lost if Colab disconnects).
3. Wait. Step 8 prints an **OVERALL** progress bar with the time left after every pass. If Colab disconnects, just **Run all** again — it picks up where it stopped.
4. At the end, `best.pt` downloads (and is in Drive under `argus_weapons/`). Put it in the project at **`ai/weapon_models/best.pt`** — the pipeline then uses it instead of YOLOE automatically.

**Data used (no accounts needed):**
* [Open Images V7](https://storage.googleapis.com/openimages/web/index.html) (Google) — real-world photos of handguns, rifles, shotguns, knives, daggers and swords, every box labelled by Google, plus *look-alike* objects (phones, drills, umbrellas, torches…) with **no** weapon labels, so the model learns they are **not** weapons (fewer false alarms).
* Test pictures come from Open Images' separate *validation* set, so the scores in step 9 are honest.

**Why version 2:** version 1 also used the Hugging Face dataset *Subh775/WeaponDetection*, but its class labels came out scrambled (its pistols were mostly turned into "long gun"), which taught the model wrong answers: handgun scored 0.25, long gun 0.29, while machete — the one class with only Google's clean labels — scored 0.79. Version 2 uses only the clean Open Images labels (`USE_HF = False` in step 3) and trains longer. It saves to a new folder (`runs/weapons_v2`), so version 1 is kept.

**"Bad footage" training (step 6b):** 40% of the training photos are turned into versions that look like CCTV / dashcam / WhatsApp footage — shrunk and blown up again (far away, low resolution), motion-blurred, too dark or washed out, noisy, heavily compressed — so the model also learns to find small, blurry weapons. Step 9 then gives two scores: on clear test photos, and on the same photos made to look like bad footage (the one closer to CIT conditions).

No public dataset has machetes, so `machete` learns from swords for now; adding a few hundred labelled machete/panga pictures later (Roboflow, free) will improve it most.

In [ ]:
# 1. Check the GPU and connect Google Drive (results are saved there)
#    Works on Google Colab AND on Kaggle (kaggle.com -> Code -> New Notebook).
import os, subprocess, sys
# Colab machines now also have a /kaggle folder, so check Kaggle's own
# setting instead - otherwise Colab results would be saved outside Drive.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB = not ON_KAGGLE
try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
except FileNotFoundError:          # CPU runtime: no nvidia-smi at all
    gpu = ""
HAS_GPU = bool(gpu.strip())
if HAS_GPU:
    print(gpu)
else:
    print("No GPU - that's fine for PREPARING the data (steps 2-6b). The prepared pictures are "
          "saved to Google Drive, so no GPU time is wasted on downloads.\n"
          "Training (step 8) needs a GPU: afterwards switch to Runtime -> Change runtime type -> "
          "T4 GPU, then Run all again - it skips the downloads.")

if ON_KAGGLE:
    print("Running on Kaggle: results are saved in /kaggle/working (the Output tab).")
else:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# 2. Install the tools (same Ultralytics version as the Argus pipeline)
#    The install updates Pillow (the image library) while Colab still has the
#    OLD one loaded - that mix causes "cannot import name '_Ink'". So after
#    installing, the session restarts ONCE to load everything fresh.
#    -> When it restarts, just press Runtime -> Run all again.
import os
from pathlib import Path
TOOLS_READY = Path("/tmp/argus_tools_installed")   # survives the restart, not a new session
if not TOOLS_READY.exists():
    !pip -q install ultralytics==8.4.174 datasets fiftyone
    TOOLS_READY.touch()
    print(">>> Tools installed. The session now restarts ONCE so the new image library loads cleanly.")
    print(">>> When Colab says the session restarted/crashed, press Runtime -> Run all AGAIN.")
    os.kill(os.getpid(), 9)
else:
    print("Tools ready (installed earlier in this session).")

In [ ]:
# 3. Settings
from pathlib import Path

# results are kept here (Colab: Google Drive; Kaggle: the notebook's Output)
DRIVE = Path("/kaggle/working/argus_weapons") if ON_KAGGLE else Path("/content/drive/MyDrive/argus_weapons")
# the training pictures (fast local disk, not kept)
DATA = Path("/tmp/weapons_data_v2") if ON_KAGGLE else Path("/content/weapons_data_v2")
RUN = "weapons_v2"                                      # results folder: argus_weapons/runs/weapons_v2
DATA_ZIP = DRIVE / "weapons_data_v2.zip"                # prepared pictures, kept in Drive between sessions

# Version 1 also used the Hugging Face dataset below, but its labels came out
# scrambled. Leave this False unless you've checked its class names yourself.
USE_HF = False
DRIVE.mkdir(parents=True, exist_ok=True)

CLASSES = ["handgun", "long_gun", "knife", "machete"]   # must match ai/weapons.py

# Hugging Face dataset label -> our class. Labels that only say "gun" /
# "weapon" (we can't tell WHICH kind) make us skip the whole picture: leaving
# such a gun unlabelled would teach the model that it is NOT a weapon.
HF_MAP = {
    "pistol": "handgun", "pistols": "handgun", "handgun": "handgun",
    "rifle": "long_gun", "shotgun": "long_gun", "long guns": "long_gun",
    "heavy gun": "long_gun", "larga": "long_gun",           # "larga" = long (gun), Spanish
    "knife": "knife", "knife_deploy": "knife", "knife_weapon": "knife",
}
HF_AMBIGUOUS = {"weapons", "weapon", "guns", "guns perspective", "heavyweapon", "al"}
# everything else (person, hand, blood, victim, ...) is simply not a weapon box

# Open Images class -> our class
OI_MAP = {
    "Handgun": "handgun",
    "Rifle": "long_gun", "Shotgun": "long_gun",
    "Knife": "knife", "Kitchen knife": "knife", "Dagger": "knife",
    "Sword": "machete",
}
OI_AMBIGUOUS = {"Weapon"}                     # Open Images' generic "Weapon" label
OI_PER_CLASS = 4000                           # max training pictures per weapon class (takes all if fewer)
OI_TEST_PER_CLASS = 400                       # test pictures per class, from Open Images' separate validation set
# Look-alikes: pictures with these but no weapon -> "this is not a weapon"
OI_LOOK_ALIKES = ["Mobile phone", "Drill (Tool)", "Hammer", "Umbrella", "Flashlight",
                  "Hair dryer", "Remote control", "Camera", "Scissors", "Spatula"]
OI_LOOK_ALIKE_TOTAL = 2000

EPOCHS = 60          # passes over the data; it stops early if it stops improving

# "Bad footage" training: this share of the training pictures is replaced by a
# version that looks like CCTV / phone / WhatsApp footage (small + blurry +
# compressed + dark or washed out), so the model also learns far, blurry guns.
DEGRADE_SHARE = 0.4
IMGSZ = 640          # picture size for training (bigger finds smaller weapons, but slower)
BATCH = 16           # pictures per step (T4 GPU)
MODEL = "yolo26s.pt" # same size as the Argus detector: good speed / accuracy balance

In [ ]:
# 4. Helpers: write pictures + YOLO label files
import random, shutil
from collections import Counter

random.seed(0)
pictures = Counter()        # split -> pictures saved this run

# Prepared pictures saved in Drive by an earlier session? Restore them and skip
# every download (steps 5, 6 and 6b then see they're done).
import zipfile
if DATA_ZIP.exists() and not (DATA / "done_degrade").exists():
    print(f"Restoring the prepared training pictures from Drive ({DATA_ZIP.stat().st_size / 1e9:.1f} GB)...")
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA)
    print("Restored - skipping all downloads.")

def yolo_line(cls_name, x, y, w, h, img_w, img_h):
    """One YOLO label line from a box given as top-left x, y + width, height
    in pixels. Returns None for boxes that are empty after clipping."""
    x1, y1 = max(0.0, x), max(0.0, y)
    x2, y2 = min(float(img_w), x + w), min(float(img_h), y + h)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    cx, cy = (x1 + x2) / 2 / img_w, (y1 + y2) / 2 / img_h
    return f"{CLASSES.index(cls_name)} {cx:.6f} {cy:.6f} {(x2 - x1) / img_w:.6f} {(y2 - y1) / img_h:.6f}"

def save_example(split, name, image, lines, source_path=None):
    """Saves one picture + its label file (empty file = no weapons in it)."""
    (DATA / split / "images").mkdir(parents=True, exist_ok=True)
    (DATA / split / "labels").mkdir(parents=True, exist_ok=True)
    img_path = DATA / split / "images" / f"{name}.jpg"
    if source_path is not None:
        shutil.copy(source_path, img_path)
    else:
        image.convert("RGB").save(img_path, quality=95)
    (DATA / split / "labels" / f"{name}.txt").write_text("\n".join(lines))
    pictures[split] += 1

In [ ]:
# 5. Hugging Face weapons dataset -> our 4 classes
from datasets import load_dataset

# The class order listed on the dataset's page (used only if the dataset
# doesn't carry its own names)
HF_NAMES_FALLBACK = ["weapons", "Aggressor", "Blood", "Guns", "Guns perspective", "Hand",
    "Heavy Gun", "Knife", "Knife_Deploy", "Knife_Weapon", "Long guns", "Person", "Pistol",
    "Rifle", "Shotgun", "Stabbing", "Victim", "al", "guns", "handgun", "heavyweapon",
    "larga", "person", "pistol", "pistols", "rifle", "shotgun", "violence", "weapon"]

def hf_class_names(split_ds):
    """The dataset's class names, read from its own metadata."""
    try:
        objects = split_ds.features["objects"]
        category = objects["category"] if isinstance(objects, dict) else objects.feature["category"]
        feature = getattr(category, "feature", category)
        return list(feature.names)
    except (AttributeError, KeyError, TypeError):
        print("Dataset has no class names stored - using the list from its web page")
        return HF_NAMES_FALLBACK

def convert_hf(split_ds, split, prefix):
    names = [n.lower().strip() for n in hf_class_names(split_ds)]
    skipped = 0
    for i, row in enumerate(split_ds):
        objs = row["objects"]
        labels = [names[c] for c in objs["category"]]
        if any(l in HF_AMBIGUOUS for l in labels):
            skipped += 1                       # unknown kind of weapon in it: skip picture
            continue
        img = row["image"]
        lines = []
        for (x, y, w, h), label in zip(objs["bbox"], labels):
            if label in HF_MAP:
                line = yolo_line(HF_MAP[label], x, y, w, h, img.width, img.height)
                if line:
                    lines.append(line)
        if lines:                              # only pictures with a weapon we know
            save_example(split, f"{prefix}{i}", img, lines)
        else:
            skipped += 1
    print(f"{prefix}: kept {pictures[split]} so far in {split}, skipped {skipped}")

if not USE_HF:
    print("Skipping the Hugging Face dataset (USE_HF = False) - see the note at the top.")
elif not (DATA / "done_hf").exists():
    hf = load_dataset("Subh775/WeaponDetection")
    print("Class names in the dataset:", hf_class_names(hf["train"]))
    convert_hf(hf["train"], "train", "hf_tr_")
    convert_hf(hf["validation"], "val", "hf_va_")
    convert_hf(hf["test"], "test", "hf_te_")
    (DATA / "done_hf").touch()

In [ ]:
# 6. Open Images: more weapon photos + look-alike objects (no weapon labels)
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone.utils.openimages import get_classes

available = set(get_classes())
weapon_classes = [c for c in OI_MAP if c in available]
look_alikes = [c for c in OI_LOOK_ALIKES if c in available]
print("Open Images weapon classes:", weapon_classes)
print("Look-alike classes:", look_alikes)

def detections_of(sample):
    """The sample's Detections field (its name depends on the FiftyOne version)."""
    for field in sample.field_names:
        value = sample[field]
        if isinstance(value, fo.Detections):
            return value.detections
    return []

SEEN = set()   # photos already used: a photo with e.g. a rifle AND a handgun comes
               # back for both searches - use it once only (else the same photo
               # could end up in both train and val, flattering the scores)

def convert_oi(dataset, prefix, weapons_only, split=None):
    """split=None: 85% train / 15% val at random; or a fixed split, e.g. "test"."""
    kept = skipped = duplicates = 0
    for sample in dataset:
        key = Path(sample.filepath).name           # Open Images photo id
        if key in SEEN:
            duplicates += 1
            continue
        dets = detections_of(sample)
        labels = [d.label for d in dets]
        if any(l in OI_AMBIGUOUS for l in labels):
            skipped += 1
            continue
        has_weapon = any(l in OI_MAP for l in labels)
        if weapons_only != has_weapon:
            skipped += 1                       # look-alike set: no weapons allowed in it
            continue
        meta = sample.metadata
        if meta is None or not meta.width:
            sample.compute_metadata()
            meta = sample.metadata
        W, H = meta.width, meta.height
        lines = []
        for d in dets:
            if d.label in OI_MAP:
                x, y, w, h = d.bounding_box     # 0-1, top-left + size
                line = yolo_line(OI_MAP[d.label], x * W, y * H, w * W, h * H, W, H)
                if line:
                    lines.append(line)
        where = split or ("val" if random.random() < 0.15 else "train")
        save_example(where, f"{prefix}{sample.id}", None, lines, source_path=sample.filepath)
        SEEN.add(key)
        kept += 1
    print(f"{prefix}: kept {kept}, skipped {skipped}, already used {duplicates}")

if not (DATA / "done_oi").exists():
    for cls in weapon_classes:
        ds = foz.load_zoo_dataset("open-images-v7", split="train", label_types=["detections"],
                                  classes=[cls], max_samples=OI_PER_CLASS, shuffle=True, seed=0, num_workers=16,
                                  dataset_name=f"oi-{cls}", drop_existing_dataset=True)
        convert_oi(ds, f"oi_{cls.replace(' ', '_')}_", weapons_only=True)
    ds = foz.load_zoo_dataset("open-images-v7", split="train", label_types=["detections"],
                              classes=look_alikes, max_samples=OI_LOOK_ALIKE_TOTAL, shuffle=True,
                              seed=0, num_workers=16, dataset_name="oi-lookalikes", drop_existing_dataset=True)
    convert_oi(ds, "oi_neg_", weapons_only=False)
    (DATA / "done_oi").touch()

# Test pictures: Open Images' separate VALIDATION set (never used for training)
if not (DATA / "done_oi_test").exists():
    for cls in weapon_classes:
        ds = foz.load_zoo_dataset("open-images-v7", split="validation", label_types=["detections"],
                                  classes=[cls], max_samples=OI_TEST_PER_CLASS, shuffle=True, seed=0, num_workers=16,
                                  dataset_name=f"oi-test-{cls}", drop_existing_dataset=True)
        convert_oi(ds, f"oitest_{cls.replace(' ', '_')}_", weapons_only=True, split="test")
    ds = foz.load_zoo_dataset("open-images-v7", split="validation", label_types=["detections"],
                              classes=look_alikes, max_samples=300, shuffle=True, seed=0, num_workers=16,
                              dataset_name="oi-test-lookalikes", drop_existing_dataset=True)
    convert_oi(ds, "oitest_neg_", weapons_only=False, split="test")
    (DATA / "done_oi_test").touch()

In [ ]:
# 6b. "Bad footage" versions: teach the model far, blurry, low-quality weapons
import cv2
import numpy as np

rng = np.random.default_rng(0)

def degrade(img):
    """Makes a clear photo look like CCTV / dashcam / phone / WhatsApp footage."""
    h, w = img.shape[:2]
    # far away / low resolution: shrink to 15-50%, then blow back up (pixelated, soft)
    s = rng.uniform(0.15, 0.5)
    small = cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
    img = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    # motion blur (half of them): a moving camera or a moving robber
    if rng.random() < 0.5:
        k = int(rng.integers(5, 15))
        kernel = np.zeros((k, k), np.float32)
        kernel[k // 2, :] = 1
        turn = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), rng.uniform(0, 180), 1)
        kernel = cv2.warpAffine(kernel, turn, (k, k))
        img = cv2.filter2D(img, -1, kernel / max(kernel.sum(), 1e-6))
    # light: too dark, too bright or washed out
    img = cv2.convertScaleAbs(img, alpha=rng.uniform(0.5, 1.3), beta=rng.uniform(-40, 40))
    # sensor noise (half of them), like night CCTV
    if rng.random() < 0.5:
        noise = rng.normal(0, rng.uniform(3, 12), img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    # heavy JPEG compression, like WhatsApp / TikTok copies
    ok, encoded = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(10, 40))])
    return cv2.imdecode(encoded, cv2.IMREAD_COLOR) if ok else img

if not (DATA / "done_degrade").exists():
    # 1) replace DEGRADE_SHARE of the training pictures (same size data = same training time)
    train_images = sorted((DATA / "train" / "images").glob("*.jpg"))
    chosen = rng.choice(len(train_images), int(len(train_images) * DEGRADE_SHARE), replace=False)
    for i in chosen:
        img = cv2.imread(str(train_images[i]))
        if img is not None:
            cv2.imwrite(str(train_images[i]), degrade(img), [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f"Training: {len(chosen)} of {len(train_images)} pictures now look like bad footage")
    # 2) a 'hard' copy of every test picture (labels unchanged: same size, same place)
    (DATA / "test_hard" / "images").mkdir(parents=True, exist_ok=True)
    (DATA / "test_hard" / "labels").mkdir(parents=True, exist_ok=True)
    for path in sorted((DATA / "test" / "images").glob("*.jpg")):
        img = cv2.imread(str(path))
        if img is None:
            continue
        cv2.imwrite(str(DATA / "test_hard" / "images" / path.name), degrade(img), [cv2.IMWRITE_JPEG_QUALITY, 95])
        shutil.copy(DATA / "test" / "labels" / (path.stem + ".txt"), DATA / "test_hard" / "labels" / (path.stem + ".txt"))
    print("Test: made a 'bad footage' copy of every test picture")
    (DATA / "done_degrade").touch()

# Save the prepared pictures to Drive once, so later sessions skip all of the above
if not DATA_ZIP.exists():
    print("Saving the prepared pictures to Google Drive (a few minutes, done only once)...")
    local_zip = DATA.parent / "weapons_data_v2.zip"
    with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:   # JPEGs: no point compressing
        for f in sorted(DATA.rglob("*")):
            if f.is_file() and not f.name.endswith(".cache"):
                z.write(f, f.relative_to(DATA))
    shutil.copy(local_zip, DATA_ZIP)
    local_zip.unlink()
    print(f"Saved: argus_weapons/{DATA_ZIP.name} ({DATA_ZIP.stat().st_size / 1e9:.1f} GB)")

In [ ]:
# 7. The training data summary + data.yaml
import yaml

# counted from the files on disk (correct even after a disconnect + re-run)
for split in ("train", "val", "test", "test_hard"):
    boxes, n = Counter(), 0
    for label_file in (DATA / split / "labels").glob("*.txt"):
        n += 1
        for line in label_file.read_text().splitlines():
            boxes[CLASSES[int(line.split()[0])]] += 1
    per_class = ", ".join(f"{c} {boxes[c]}" for c in CLASSES)
    print(f"{split:5}: {n:6} pictures | weapon boxes: {per_class}")

data_yaml = DATA / "data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(DATA), "train": "train/images", "val": "val/images", "test": "test/images",
    "names": {i: n for i, n in enumerate(CLASSES)},
}))
print(data_yaml.read_text())

# the same test pictures, but made to look like bad footage (step 9 scores both)
hard_yaml = DATA / "data_hard.yaml"
hard_yaml.write_text(yaml.safe_dump({
    "path": str(DATA), "train": "train/images", "val": "val/images", "test": "test_hard/images",
    "names": {i: n for i, n in enumerate(CLASSES)},
}))

In [ ]:
# 8. Train (resumes automatically after a disconnect)
from ultralytics import YOLO
import time
import torch

def overall_progress(trainer):
    """After each pass over the data: one overall progress bar with time left.
    (The bars Ultralytics prints are for ONE pass only.)"""
    if getattr(overall_progress, "finished", False):
        return                          # Ultralytics' final check after the last pass
    done = min(trainer.epoch + 1, trainer.epochs)
    total = trainer.epochs
    passes_this_session = done - trainer.start_epoch
    per_pass = (time.time() - trainer.train_time_start) / max(1, passes_this_session)
    left_min = round(per_pass * (total - done) / 60)
    filled = round(30 * done / total)
    score = (trainer.metrics or {}).get("metrics/mAP50(B)")
    score_text = f" | score so far (mAP50): {score:.2f}" if score is not None else ""
    print(f"\nOVERALL [{'#' * filled}{'-' * (30 - filled)}] pass {done}/{total} "
          f"| about {left_min // 60} h {left_min % 60} min left{score_text}"
          f"\n(it may finish earlier: it stops by itself once it stops improving)\n")
    if trainer.stop or done >= total:
        overall_progress.finished = True
        print("OVERALL: training finished - now checking the best model...\n")

last = DRIVE / "runs" / RUN / "weights" / "last.pt"
finished = last.exists() and torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1
if not finished and not HAS_GPU:
    raise RuntimeError("DATA IS READY and saved in Google Drive. Training needs a GPU: "
                       "Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all "
                       "(it restores the pictures from Drive and skips the downloads).")
if finished:
    print("Training already finished - skipping to the results.")
    print(f"(To train again from scratch, delete argus_weapons/runs/{RUN} in Google Drive.)")
elif last.exists():
    print("Resuming from", last)
    model = YOLO(str(last))
    model.add_callback("on_fit_epoch_end", overall_progress)
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.add_callback("on_fit_epoch_end", overall_progress)
    model.train(
        data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=20,                      # stop early if no better for 20 passes
        project=str(DRIVE / "runs"), name=RUN, exist_ok=True,
        save_period=5,                    # extra safety copies in Drive
        fliplr=0.5, degrees=5, scale=0.6, # pictures flipped / turned / zoomed in and out
        close_mosaic=10, seed=0,
    )

In [ ]:
# 9. How good is it? Tested on pictures it never trained on.
best = DRIVE / "runs" / RUN / "weights" / "best.pt"
model = YOLO(str(best))
def score_table(metrics, title):
    print(f"\n{title}")
    tested = list(metrics.box.ap_class_index)    # classes that had test pictures
    for i, name in enumerate(CLASSES):
        if i in tested:
            p, r, ap50, ap = metrics.box.class_result(tested.index(i))
            print(f"  {name:9} precision {p:.2f}  recall {r:.2f}  mAP50 {ap50:.2f}")
        else:
            print(f"  {name:9} (no test pictures of this class)")
    print(f"  overall   mAP50 {metrics.box.map50:.2f}")

clean = model.val(data=str(data_yaml), split="test", imgsz=IMGSZ, batch=BATCH)
hard = model.val(data=str(DATA / "data_hard.yaml"), split="test", imgsz=IMGSZ, batch=BATCH)
print("\nmAP50 = how well it finds each kind (0-1, higher is better; 0.7+ is good)")
print("precision = when it says 'weapon', how often it's right;"
      " recall = of all weapons, how many it found")
score_table(clean, "CLEAR test photos:")
score_table(hard, "Same photos made to look like BAD FOOTAGE (small, blurry, compressed) - closer to CIT footage:")

In [ ]:
# 10. Save best.pt for the Argus pipeline
shutil.copy(best, DRIVE / "best.pt")
print("Put it in the project as:  ai/weapon_models/best.pt")
if ON_KAGGLE:
    print("Kaggle: download it from the Output panel on the right: argus_weapons/best.pt")
else:
    print("Saved to Google Drive: argus_weapons/best.pt")
    from google.colab import files
    files.download(str(DRIVE / "best.pt"))